# Validations Framework

In [0]:
from src.validations import (
    validate_no_nulls,
    validate_no_duplicates,
    validate_referential_integrity,
    validate_row_count,
    validate_required_columns
)

print("Validation framework imported successfully")

In [0]:
dim_facility_df = spark.table(
    "databricks_project1.gold.dim_facility"
)

dim_labor_df = spark.table(
    "databricks_project1.gold.dim_labor_position"
)

dim_employee_df = spark.table(
    "databricks_project1.gold.dim_employee"
)

fact_df = spark.table(
    "databricks_project1.gold.fact_employee_payroll"
)

print("Gold tables loaded successfully")

In [0]:
print("Dim Facility:", dim_facility_df.count())
print("Dim Labor Position:", dim_labor_df.count())
print("Dim Employee:", dim_employee_df.count())
print("Fact Employee Payroll:", fact_df.count())

## Row-count validation

In [0]:
print("Dim Facility:")
print(validate_row_count(dim_facility_df, 57))

print("\nDim Labor Position:")
print(validate_row_count(dim_labor_df, 308))

print("\nDim Employee:")
print(validate_row_count(dim_employee_df, 651))

print("\nFact Employee Payroll:")
print(validate_row_count(fact_df, 651))

## NULL validation

In [0]:
print("Dim Facility NULL validation:")
print(
    validate_no_nulls(
        dim_facility_df,
        ["Facility_Code", "Facility_Name"]
    )
)

print("\nDim Labor Position NULL validation:")

dim_labor_nulls = validate_no_nulls(
    dim_labor_df,
    ["Labor_Position_Code", "Labor_Position_Desc"]
)

print(dim_labor_nulls)

print("\nDim Employee NULL validation:")
print(
    validate_no_nulls(
        dim_employee_df,
        ["EmployeeKey", "Employee_Code"]
    )
)

print("\nFact Employee Payroll NULL validation:")
print(
    validate_no_nulls(
        fact_df,
        [
            "EmployeeKey",
            "Employee_Code",
            "Facility_Code",
            "Labor_Position_Code"
        ]
    )
)

## Duplicate validation

In [0]:
print("Dim Facility duplicates:")
print(
    validate_no_duplicates(
        dim_facility_df,
        ["Facility_Code"]
    )
)

print("\nDim Labor Position duplicates:")
print(
    validate_no_duplicates(
        dim_labor_df,
        ["Labor_Position_Code"]
    )
)

print("\nDim Employee duplicates:")
print(
    validate_no_duplicates(
        dim_employee_df,
        ["EmployeeKey"]
    )
)

print("\nFact Employee Payroll duplicates:")
print(
    validate_no_duplicates(
        fact_df,
        ["EmployeeKey"]
    )
)

## Referential integrity

In [0]:
print("\n===== GOLD REFERENTIAL INTEGRITY =====")

fact_employee_ri = validate_referential_integrity(
    fact_df,
    dim_employee_df,
    "EmployeeKey",
    "EmployeeKey"
)

fact_facility_ri = validate_referential_integrity(
    fact_df,
    dim_facility_df,
    "Facility_Code",
    "Facility_Code"
)

fact_labor_ri = validate_referential_integrity(
    fact_df,
    dim_labor_df,
    "Labor_Position_Code",
    "Labor_Position_Code"
)

print("Fact → Dim Employee", fact_employee_ri)
print("Fact → Dim Facility", fact_facility_ri)
print("Fact → Dim Labor Position", fact_labor_ri)

## Required-column validation

In [0]:
print("Dim Facility:")
print(
    validate_required_columns(
        dim_facility_df,
        [
            "Facility_Code",
            "Facility_Name"
        ]
    )
)

print("\nDim Labor Position:")
print(
    validate_required_columns(
        dim_labor_df,
        [
            "Labor_Position_Code",
            "Labor_Position_Desc",
            "ingestion_timestamp",
            "source_file"
        ]
    )
)

print("\nDim Employee:")
print(
    validate_required_columns(
        dim_employee_df,
        [
            "EmployeeKey",
            "Employee_Code",
            "Employee_Status",
            "Facility_Code",
            "Labor_Position_Code",
            "StartDate",
            "EndDate",
            "IsCurrent",
            "ingestion_timestamp",
            "source_file"
        ]
    )
)

print("\nFact Employee Payroll:")
print(
    validate_required_columns(
        fact_df,
        [
            "EmployeeKey",
            "Employee_Code",
            "Employee_Status",
            "Facility_Code",
            "Labor_Position_Code",
            "Labor_Position_Desc",
            "Birth_Date",
            "Hire_Date",
            "Rehire_Date",
            "Termination_Date",
            "ingestion_timestamp",
            "source_file"
        ]
    )
)

In [0]:
print("\n===== DQ ISSUE SUMMARY =====")

if not fact_labor_ri["passed"]:
    print(
        "FAIL: Fact → Dim Labor Position has "
        f"{fact_labor_ri['missing_keys']} missing key(s)."
    )

if not dim_labor_nulls["Labor_Position_Desc"]["passed"]:
    print(
        "FAIL: Dim Labor Position contains "
        f"{dim_labor_nulls['Labor_Position_Desc']['null_count']} "
        "NULL Labor_Position_Desc value(s)."
    )

# Logger framework

In [0]:
from src.logger import (
    get_logger,
    log_pipeline_start,
    log_pipeline_end,
    log_row_count,
    log_error
)

print("Logger framework imported successfully")

In [0]:
import time

logger = get_logger("GoldDQTest")

start_time = time.time()

log_pipeline_start(
    logger,
    "Gold DQ Test"
)

# -----------------------------
# Run DQ validations
# -----------------------------

dim_facility_row_count = validate_row_count(
    dim_facility_df,
    57
)

dim_labor_row_count = validate_row_count(
    dim_labor_df,
    308
)

dim_employee_row_count = validate_row_count(
    dim_employee_df,
    651
)

fact_row_count = validate_row_count(
    fact_df,
    651
)

fact_employee_ri = validate_referential_integrity(
    fact_df,
    dim_employee_df,
    "EmployeeKey",
    "EmployeeKey"
)

fact_facility_ri = validate_referential_integrity(
    fact_df,
    dim_facility_df,
    "Facility_Code",
    "Facility_Code"
)

fact_labor_ri = validate_referential_integrity(
    fact_df,
    dim_labor_df,
    "Labor_Position_Code",
    "Labor_Position_Code"
)

# -----------------------------
# Overall DQ status
# -----------------------------

overall_passed = all([
    dim_facility_row_count["passed"],
    dim_labor_row_count["passed"],
    dim_employee_row_count["passed"],
    fact_row_count["passed"],
    fact_employee_ri["passed"],
    fact_facility_ri["passed"],
    fact_labor_ri["passed"]
])

status = "SUCCESS" if overall_passed else "FAILED"

print("Overall DQ Status:", status)

# -----------------------------
# Log row counts
# -----------------------------

log_row_count(
    logger,
    "DimFacility",
    dim_facility_row_count["actual_count"]
)

log_row_count(
    logger,
    "DimLaborPosition",
    dim_labor_row_count["actual_count"]
)

log_row_count(
    logger,
    "DimEmployee",
    dim_employee_row_count["actual_count"]
)

log_row_count(
    logger,
    "FactEmployeePayroll",
    fact_row_count["actual_count"]
)

# -----------------------------
# Log pipeline completion
# -----------------------------

log_pipeline_end(
    logger,
    "Gold DQ Test",
    start_time,
    status=status
)

print("DQ logging completed")

# Incremental Pipeline

In [0]:
from pyspark.sql import functions as F

spark.sql("""
CREATE TABLE IF NOT EXISTS databricks_project1.silver.pipeline_watermark (
    pipeline_name STRING,
    source_name STRING,
    last_processed_timestamp TIMESTAMP,
    updated_at TIMESTAMP
)
USING DELTA
""")

print("Watermark table ready")

### Add reusable watermark functions

In [0]:
from pyspark.sql import functions as F


def get_watermark(
    pipeline_name,
    source_name
):
    """
    Return the last processed timestamp for a pipeline/source.
    Returns None if no watermark exists.
    """

    watermark_df = spark.table(
        "databricks_project1.silver.pipeline_watermark"
    )

    result = (
        watermark_df
        .filter(
            (F.col("pipeline_name") == pipeline_name) &
            (F.col("source_name") == source_name)
        )
        .select("last_processed_timestamp")
        .limit(1)
        .collect()
    )

    if not result:
        return None

    return result[0]["last_processed_timestamp"]


def update_watermark(
    pipeline_name,
    source_name,
    processed_timestamp
):
    """
    Insert or update the watermark for a pipeline/source.
    """

    spark.sql(f"""
        MERGE INTO databricks_project1.silver.pipeline_watermark AS target
        USING (
            SELECT
                '{pipeline_name}' AS pipeline_name,
                '{source_name}' AS source_name,
                TIMESTAMP('{processed_timestamp}') AS last_processed_timestamp,
                current_timestamp() AS updated_at
        ) AS source
        ON target.pipeline_name = source.pipeline_name
        AND target.source_name = source.source_name

        WHEN MATCHED THEN UPDATE SET
            target.last_processed_timestamp =
                source.last_processed_timestamp,
            target.updated_at =
                source.updated_at

        WHEN NOT MATCHED THEN INSERT (
            pipeline_name,
            source_name,
            last_processed_timestamp,
            updated_at
        )
        VALUES (
            source.pipeline_name,
            source.source_name,
            source.last_processed_timestamp,
            source.updated_at
        )
    """)

### Test get_watermark()

In [0]:
watermark = get_watermark(
    "Silver Employee",
    "Employee_Payroll.xlsx"
)

print("Initial watermark:", watermark)

### Test update_watermark()

In [0]:
from datetime import datetime

test_timestamp = datetime(2026, 8, 14, 6, 0, 0)

update_watermark(
    "Silver Employee",
    "Employee_Payroll.xlsx",
    test_timestamp
)

print("Watermark updated successfully")

In [0]:
display(
    spark.table(
        "databricks_project1.silver.pipeline_watermark"
    )
)

### Verify get_watermark()

In [0]:
current_watermark = get_watermark(
    "Silver Employee",
    "Employee_Payroll.xlsx"
)

print("Retrieved watermark:", current_watermark)

In [0]:
import importlib
import src.watermark

importlib.reload(src.watermark)

from src.watermark import (
    get_watermark,
    update_watermark
)

print("Watermark framework imported successfully")

In [0]:
current_watermark = get_watermark(
    "Silver Employee",
    "Employee_Payroll.xlsx"
)

print("Watermark from src:", current_watermark)